In [1]:
import duckdb
con = duckdb.connect("olist.duckdb")
print("connected")

connected


In [2]:
for t in ["order_level", "item_level", "delivery_orders", "delivery_overall",
          "delivery_status_summary", "repeat_summary", "category_watchlist",
          "overall_freight", "seller_impact"]:
    print(t, con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0], "rows")

order_level 99441 rows
item_level 112650 rows
delivery_orders 96470 rows
delivery_overall 1 rows
delivery_status_summary 2 rows
repeat_summary 1 rows
category_watchlist 12 rows
overall_freight 1 rows
seller_impact 1 rows


In [3]:
con.execute("""
CREATE OR REPLACE TABLE baseline AS
SELECT COUNT(DISTINCT customer_unique_id) AS customers,
       COUNT(*) AS orders,
       1.0 * COUNT(*) / COUNT(DISTINCT customer_unique_id) AS orders_per_customer,
       SUM(item_revenue) AS revenue,
       SUM(item_revenue) / COUNT(*) AS aov
FROM order_level
WHERE order_status = 'delivered' AND item_revenue IS NOT NULL
""")

con.execute("""
SELECT customers,
       orders,
       ROUND(orders_per_customer, 4) AS orders_per_customer,
       ROUND(aov, 2) AS aov,
       ROUND(revenue, 2) AS revenue,
       ROUND(customers * orders_per_customer * aov, 2) AS customers_x_opc_x_aov,
       (SELECT ROUND(SUM(price), 2) FROM item_level WHERE order_status = 'delivered') AS day8_delivered_revenue
FROM baseline
""").df()

,customers,orders,orders_per_customer,aov,revenue,customers_x_opc_x_aov,day8_delivered_revenue
0,93358,96478,1.0334,137.04,13221498.11,13221498.11,13221498.11


In [4]:
con.execute("""
CREATE OR REPLACE TABLE first_order_repeat AS
WITH o AS (
  SELECT customer_unique_id, order_id,
         ROW_NUMBER() OVER (PARTITION BY customer_unique_id
                            ORDER BY order_purchase_timestamp, order_id) AS rn,
         COUNT(*) OVER (PARTITION BY customer_unique_id) AS n_orders
  FROM order_level
  WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL
),
f AS (
  SELECT o.customer_unique_id, o.n_orders, d.delivery_status
  FROM o
  JOIN delivery_orders d USING (order_id)
  WHERE o.rn = 1
)
SELECT delivery_status,
       COUNT(*) AS customers,
       SUM(CASE WHEN n_orders >= 2 THEN 1 ELSE 0 END) AS repeaters,
       ROUND(100.0 * SUM(CASE WHEN n_orders >= 2 THEN 1 ELSE 0 END) / COUNT(*), 2) AS repeat_pct
FROM f
GROUP BY 1
ORDER BY 1
""")

con.execute("SELECT * FROM first_order_repeat").df()

,delivery_status,customers,repeaters,repeat_pct
0,Late,6355,160.0,2.52
1,On time,86995,2641.0,3.04


In [5]:
import pandas as pd

In [6]:
# ---------------- ASSUMPTIONS (scenario inputs, not measurements) ----------------
LATE_REDUCTION   = 0.33   # late deliveries cut by one third
REPEAT_UPLIFT_PP = 1.0    # repeat-purchase rate rises by 1 percentage point
FREIGHT_CUT      = 0.10   # freight cut by 10% in the watch-list categories
ELASTICITY       = 0.5    # illustrative: 1% lower total price paid -> 0.5% more sales
# ---------------------------------------------------------------------------------

b   = con.execute("SELECT * FROM baseline").df().iloc[0]
r   = con.execute("SELECT * FROM repeat_summary").df().iloc[0]
d   = con.execute("SELECT * FROM delivery_overall").df().iloc[0]
s   = con.execute("SELECT * FROM delivery_status_summary").df().set_index("delivery_status")
fr  = con.execute("SELECT * FROM first_order_repeat").df().set_index("delivery_status")
w   = con.execute("SELECT * FROM category_watchlist").df()

In [7]:
# extra orders a repeat customer places, measured from Day 6
extra_orders_per_repeater = r.customers * (r.avg_orders_per_customer - 1) / r.repeat_customers

In [8]:
# Lever 1: fewer late deliveries
gap_pp = fr.loc["On time", "repeat_pct"] - fr.loc["Late", "repeat_pct"]
late_first_customers = fr.loc["Late", "customers"]
extra_repeaters_1 = late_first_customers * LATE_REDUCTION * max(gap_pp, 0) / 100
rev1 = extra_repeaters_1 * extra_orders_per_repeater * b.aov
low_gap = s.loc["Late", "low_review_pct"] - s.loc["On time", "low_review_pct"]
low_avoided = d.late_orders * LATE_REDUCTION * low_gap / 100

In [9]:
# Lever 2: higher repeat purchase
extra_repeaters_2 = b.customers * REPEAT_UPLIFT_PP / 100
rev2 = extra_repeaters_2 * extra_orders_per_repeater * b.aov

In [10]:
# Lever 3: lower freight in large, freight-heavy categories
if len(w) > 0:
    wl_rev, wl_fr = w.revenue.sum(), w.freight.sum()
    saved = wl_fr * FREIGHT_CUT
    price_drop = saved / (wl_rev + wl_fr)
    rev3 = wl_rev * price_drop * ELASTICITY
else:
    wl_rev = wl_fr = saved = price_drop = rev3 = 0.0

rows = [
    ("1. Reduce late deliveries", f"Late deliveries cut by {LATE_REDUCTION:.0%}", rev1),
    ("2. Raise repeat purchase", f"Repeat rate up {REPEAT_UPLIFT_PP} percentage point", rev2),
    ("3. Cut freight in high-freight categories",
     f"Freight down {FREIGHT_CUT:.0%} on {len(w)} watch-list categories, elasticity {ELASTICITY}", rev3),
]
lev = pd.DataFrame(rows, columns=["lever", "assumption", "effect_base"])
lev["effect_low"] = lev.effect_base * 0.5
lev["effect_high"] = lev.effect_base * 2
lev["pct_of_revenue_base"] = 100 * lev.effect_base / b.revenue
lev = lev.round(2).sort_values("effect_base", ascending=False)
lev.to_csv("dashboard/levers.csv", index=False)

In [11]:
print("Repeat rate gap (on-time minus late first order), percentage points:", round(gap_pp, 2))
print("Extra orders per repeat customer:", round(extra_orders_per_repeater, 3))
print("1-2 star orders avoided by lever 1:", round(low_avoided))
lev

Repeat rate gap (on-time minus late first order), percentage points: 0.52
Extra orders per repeat customer: 1.1
1-2 star orders avoided by lever 1: 1146


,lever,assumption,effect_base,effect_low,effect_high,pct_of_revenue_base
1,2. Raise repeat purchase,Repeat rate up 1.0 percentage point,140720.15,70360.07,281440.30,1.06
2,3. Cut freight in high-freight categories,"Freight down 10% on 12 watch-list categories, ...",45460.87,22730.43,90921.73,0.34
0,1. Reduce late deliveries,Late deliveries cut by 33%,1643.76,821.88,3287.51,0.01


In [12]:
FENCE = "`" * 3
ov  = con.execute("SELECT * FROM overall_freight").df().iloc[0]
imp = con.execute("SELECT * FROM seller_impact").df().iloc[0]

def n0(x):
    return f"{x:,.0f}"

# Lever 1 wording depends on what the data shows
if gap_pp > 0:
    l1_effect = (f"If {LATE_REDUCTION:.0%} of late first orders arrived on time, about {n0(extra_repeaters_1)} "
                 f"more customers would reorder, worth about {n0(rev1)} in revenue. "
                 f"It would also avoid roughly {n0(low_avoided)} one- or two-star reviews.")
else:
    l1_effect = (f"The data shows no repeat-purchase gap between late and on-time first orders, so no revenue effect is claimed. "
                 f"The benefit is mainly about {n0(low_avoided)} fewer one- or two-star reviews.")

if imp.flagged_sellers > 0:
    seller_line = (f"{int(imp.flagged_sellers)} flagged sellers account for "
                   f"{round(100 * imp.flagged_late_orders / imp.all_late_orders, 1)}% of all late orders, "
                   f"so seller coaching is the practical way to pull this lever.")
else:
    seller_line = "No seller met the flag rule, so the late orders are spread across many sellers and regions."

table_rows = "\n".join(
    f"| {x.lever} | {x.assumption} | {n0(x.effect_low)} | {n0(x.effect_base)} | {n0(x.effect_high)} | {x.pct_of_revenue_base:.1f}% |"
    for x in lev.itertuples()
)

md = f"""# Revenue levers and assumptions

## 1. Metric tree

Revenue = Customers x Orders per customer x Average order value (delivered orders only).

{FENCE}mermaid
flowchart TD
  R["Revenue<br/>{n0(b.revenue)}"] --> C["Customers<br/>{n0(b.customers)}"]
  R --> O["Orders per customer<br/>{b.orders_per_customer:.3f}"]
  R --> A["Average order value<br/>{b.aov:.2f}"]
  C --> C1["New customers acquired"]
  C --> C2["Checkout conversion: price plus freight<br/>freight is {ov.freight_pct_of_price}% of price (LEVER 3)"]
  O --> O1["Repeat purchase rate<br/>{r.repeat_pct}% (LEVER 2)"]
  O --> O2["Delivery experience and reviews<br/>late rate {d.late_pct}% (LEVER 1)"]
  A --> A1["Category mix and items per order"]
  classDef lever fill:#ffe8a3,stroke:#b8860b
  class C2,O1,O2 lever
{FENCE}

Baseline from the data: {n0(b.customers)} customers x {b.orders_per_customer:.3f} orders per customer x {b.aov:.2f} average order value = {n0(b.revenue)} revenue.

## 2. Summary of the three levers

Effects are in the same currency units as the dataset and cover the whole period of the data (about two years). Low is half the assumed change and high is double it.

| Lever | Assumption | Low | Base | High | Base as % of revenue |
|---|---|---|---|---|---|
{table_rows}

## 3. Lever details

### Lever 1: Reduce late deliveries
- **Measured:** {d.late_pct}% of delivered orders arrive late. Late orders average a review of {s.loc['Late','avg_review']} against {s.loc['On time','avg_review']} for on-time orders, and {s.loc['Late','low_review_pct']}% of them get 1-2 stars against {s.loc['On time','low_review_pct']}%.
- **Measured:** customers whose first order was late reorder at {fr.loc['Late','repeat_pct']}%, against {fr.loc['On time','repeat_pct']}% when the first order was on time.
- **Assumption:** late deliveries fall by {LATE_REDUCTION:.0%}, for example through more realistic delivery estimates and coaching of the worst sellers.
- **Estimate:** {l1_effect}
- **Action path:** {seller_line}
- **Caveat:** this is an association; late delivery may not be the only reason these customers reorder less.

### Lever 2: Raise repeat purchase
- **Measured:** {r.repeat_pct}% of customers order more than once, with {r.avg_orders_per_customer} orders per customer on average. A repeat customer places about {extra_orders_per_repeater:.2f} extra orders.
- **Assumption:** a retention programme lifts the repeat rate by {REPEAT_UPLIFT_PP} percentage point, which is {n0(extra_repeaters_2)} extra repeat customers.
- **Estimate:** about {n0(rev2)} in revenue at the average order value of {b.aov:.2f}.
- **Caveat:** the size of the uplift is an assumption; it should be tested with a real experiment.

### Lever 3: Cut freight cost in high-freight categories
- **Measured:** freight is {ov.freight_pct_of_price}% of item price overall. {len(w)} large categories sit above that level, with {n0(wl_rev)} in revenue and {n0(wl_fr)} in freight.
- **Assumption:** freight in those categories falls by {FREIGHT_CUT:.0%} (for example through carrier renegotiation or a free-shipping threshold), and sales respond with an elasticity of {ELASTICITY}.
- **Estimate:** customers would pay about {n0(saved)} less in freight, a {100 * price_drop:.2f}% drop in total price paid, which would add about {n0(rev3)} in revenue.
- **Caveat:** the elasticity is illustrative, not measured, and who pays for the freight cut is not modelled.

## 4. Limits
- The data has no product costs, so these are revenue effects, not profit effects.
- Levers 2 and 3 rest on assumptions, not on measured responses. Lever 1 uses a measured association.
- Effects overlap, so the three should not simply be added together.
- Late-first-order customers reorder less, but the data cannot prove lateness is the cause.
"""

with open("memo/levers.md", "w", encoding="utf-8") as fh:
    fh.write(md)
print("written memo/levers.md")

written memo/levers.md
